Load Packages

In [5]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

Load data from directory 

In [6]:
file_path = Path("Data/raw/preprocessed_capstone2025.json")
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

Define constants and variables

In [7]:
# Replacements for malformed themes
rthema_corrections = {
    "HandelsrechtGesellschaftsrecht": "Handelsrecht Gesellschaftsrecht",
    "MietePacht": "Miete Pacht"
}

allowed_rthemen = {"Zivilverfahrensrecht", "SchuldrechtAT", "Schadensersatz", "Handelsrecht Gesellschaftsrecht", "BGBAT", "Miete Pacht", "Sachenrecht", "Versicherungsrecht","Kauf Tausch Leasing","Erbschaft Schenkung", "Schuldverhältnisse", "Werkvertrag", "EU-Recht", "Wohnungseigentum", "Sonstiges Recht", "Reisevertrag"}
date_cutoff = "2000-01-01"

json_output = "Data/clean/filtered_output.json"

Save as JSON

In [8]:
# Function to save the cleanded data to JSON, with helper functions

def correct_rthema_list(rthema_list):
    """Apply correction to rthema list items."""
    if not isinstance(rthema_list, list):
        return rthema_list
    return [rthema_corrections.get(r, r) for r in rthema_list]

def rthema_is_valid(rthema_list):
    """Check if corrected rthema is entirely within the allowed set."""
    if not rthema_list or not isinstance(rthema_list, list):
        return False
    return set(rthema_list).issubset(allowed_rthemen)

# Process and filter in-place
filtered_data = {}

for doc_id, content in data.items():
    bibliography = content.get("bibliographische-angaben", {})
    text = content.get("text", {})
    allgemeine = content.get("allgemeine-angaben", {})

    datum_str = bibliography.get("datum", [None])[0]
    try:
        datum = pd.to_datetime(datum_str, format="%Y-%m-%d", errors="coerce")
    except Exception:
        datum = pd.NaT

    rthema_corrected = correct_rthema_list(allgemeine.get("rthema", []))

    if (
        content.get("dokument_art", [None])[0] == "Urteil" and
        content.get("dokument_typ", [None])[0] == "Obere Rechtsprechung" and
        bibliography.get("institution", [None])[0] is not None and
        datum and datum > pd.Timestamp(date_cutoff) and
        rthema_is_valid(rthema_corrected)
    ):
        # Save corrected rthema back into the content
        allgemeine["rthema"] = rthema_corrected
        content["bibliographische-angaben"]["datum"] = [datum.strftime("%Y-%m-%d")]
        filtered_data[doc_id] = content

# Save output JSON
with open(json_output, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)